In [ ]:
import pandas as pd
import json
import os
import numpy as np
from datasets import Dataset

from transformers import (
    set_seed,
    TrainingArguments, 
    Trainer,
    DataCollatorForLanguageModeling,
    AutoModelForMaskedLM,
    AutoTokenizer,
)

from time import time
import pickle
import matplotlib.pyplot as plt
import random
from tqdm import tqdm
from unsloth import FastLanguageModel
import torch

def_seed = 42

set_seed(def_seed)
np.random.seed(def_seed)
import random
random.seed(def_seed)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

from sklearn.model_selection import train_test_split


/tmp/ipykernel_49298/2222613534.py:16: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 10-18 05:12:54 [__init__.py:244] Automatically detected platform cuda.


Loading synthetic data for labeling: to preserve anonymity, the links to Hugging Face have been removed, but a portion of 10,000 samples from each dataset is included.

In [ ]:

def pretrain_Bert(
        df=None,
        base=None,
        tokenizer=None,
        save_path=None,
):
    
    df = df.sample(frac=1, random_state=def_seed).reset_index(drop=True)

    dataset = Dataset.from_pandas(df)
    
    def tokenize_and_chunk(example):
        outputs = tokenizer(
            example["text"],
            truncation=True,
            max_length=512,
            stride=0,  
        )

        result = {
            "input_ids": outputs["input_ids"],
            "attention_mask": outputs["attention_mask"],
        }

        return result


    tokenized_dataset = dataset.map(
        tokenize_and_chunk,
        batched=True,
        remove_columns=["text"],  
        num_proc=10,
    )
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=0.15,
    )

    training_args = TrainingArguments(
        output_dir=save_path,
        overwrite_output_dir=True,
        num_train_epochs=1,
        save_strategy="steps",        
        save_steps=3000,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=1,
        learning_rate=1e-5,
        lr_scheduler_type="cosine",
        warmup_ratio=0.2,
        max_grad_norm=0.2,
        logging_steps=200,
        fp16=False,
        bf16=True,
        gradient_accumulation_steps=2,
        gradient_checkpointing=True,
        report_to="none",              
        eval_steps=5000,
        max_steps=-1,                  
        log_level="debug",
        save_total_limit=10,
        dataloader_num_workers=16,
    )



    trainer = Trainer(
        model=base,
        args=training_args,
        train_dataset=tokenized_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,

    )
    # trainer.train()
    trainer.train(resume_from_checkpoint=True)
    return trainer

In [ ]:
datataset_4L_2_7M = load_dataset("anonymousOWSHateLLM/2_7M_texts")
datataset_Eng = load_dataset("anonymousOWSHateLLM/eng_unlabel")
datataset_Deu = load_dataset("anonymousOWSHateLLM/deu_unlabel")
datataset_Spa = load_dataset("anonymousOWSHateLLM/span_unlabel")


In [ ]:
df = pd.DataFrame("datataset_4L_2_7M")

In [ ]:
model_name = "google-bert/bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

In [ ]:
trainer = pretrain_Bert(
    df=df,
    base=model,
    tokenizer=tokenizer,
    save_path="Ows4L"
)